# ResNet50 with ImageNet Pretraining — Bone Tumor Classification

This notebook contains the ResNet50 + ImageNet pretrained model pipeline for binary classification of bone X-ray images into **Normal (0)** and **Cancer/Tumor (1)**.

The ResNet50 backbone is frozen and only the final fully connected layer is trained. The best checkpoint is selected using validation accuracy and evaluated once on the held-out test set.


In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from PIL import Image
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    roc_curve
)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
import shutil

DRIVE_DATA = Path("/content/drive/MyDrive/bone_tumor_data/final")
LOCAL_DATA = Path("/content/bone_tumor_data/final")

print("Copying dataset from Google Drive to Colab local storage...")

if LOCAL_DATA.exists():
    shutil.rmtree(LOCAL_DATA)

shutil.copytree(DRIVE_DATA, LOCAL_DATA)

print("Dataset copied successfully.")
print("Local path:", LOCAL_DATA)
print("Exists:", LOCAL_DATA.exists())


In [ ]:
DATA_ROOT = Path("/content/bone_tumor_data/final")

print("Dataset path:", DATA_ROOT)
print("Dataset exists:", DATA_ROOT.exists())

for split in ["train", "valid", "test"]:
    print(f"{split.upper()}:")
    print("  Images:", (DATA_ROOT / split / "images").exists())
    print("  Metadata:", (DATA_ROOT / split / "metadata.csv").exists())


In [ ]:
for split in ["train", "valid", "test"]:
    image_path = DATA_ROOT / split / "images"
    metadata_path = DATA_ROOT / split / "metadata.csv"

    image_count = len(list(image_path.glob("*.png")))
    metadata_count = len(pd.read_csv(metadata_path))

    print(f"{split.upper()}:")
    print("  PNG images:", image_count)
    print("  Metadata rows:", metadata_count)


In [ ]:
bad_files = []

for split in ["train", "valid", "test"]:
    metadata = pd.read_csv(DATA_ROOT / split / "metadata.csv")
    image_dir = DATA_ROOT / split / "images"

    for filename in metadata["filename"]:
        image_path = image_dir / filename

        try:
            with Image.open(image_path) as img:
                img.verify()
        except Exception as e:
            bad_files.append({
                "split": split,
                "filename": filename,
                "error": str(e)
            })

print("Unreadable/missing images:", len(bad_files))

if bad_files:
    display(pd.DataFrame(bad_files))


In [ ]:
for split in ["train", "valid", "test"]:
    metadata = pd.read_csv(DATA_ROOT / split / "metadata.csv")

    print(f"\n{split.upper()}")
    print(metadata["cancer"].value_counts().sort_index())


In [ ]:
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

valid_test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("Training preprocessing created.")
print("Validation/Test preprocessing created.")


In [ ]:
class BoneTumorDataset(Dataset):
    def __init__(self, split, transform=None):
        self.split = split
        self.transform = transform
        self.image_dir = DATA_ROOT / split / "images"

        metadata = pd.read_csv(DATA_ROOT / split / "metadata.csv")

        self.metadata = metadata[
            metadata["filename"].apply(
                lambda x: (self.image_dir / x).exists()
            )
        ].reset_index(drop=True)

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, index):
        row = self.metadata.iloc[index]

        image_path = self.image_dir / row["filename"]
        image = Image.open(image_path).convert("RGB")
        label = int(row["cancer"])

        if self.transform:
            image = self.transform(image)

        return image, label

print("BoneTumorDataset class created successfully.")


In [ ]:
train_dataset = BoneTumorDataset(
    split="train",
    transform=train_transform
)

valid_dataset = BoneTumorDataset(
    split="valid",
    transform=valid_test_transform
)

test_dataset = BoneTumorDataset(
    split="test",
    transform=valid_test_transform
)

print("Train:", len(train_dataset))
print("Validation:", len(valid_dataset))
print("Test:", len(test_dataset))


In [ ]:
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("DataLoaders created successfully.")
print("Batch size:", BATCH_SIZE)
print("Training batches:", len(train_loader))
print("Validation batches:", len(valid_loader))
print("Test batches:", len(test_loader))


In [ ]:
images, labels = next(iter(train_loader))

print("Images shape:", images.shape)
print("Labels shape:", labels.shape)
print("Unique labels:", labels.unique().tolist())


In [ ]:
model = models.resnet50(
    weights=models.ResNet50_Weights.DEFAULT
)

model.fc = nn.Linear(
    in_features=2048,
    out_features=2
)

for param in model.parameters():
    param.requires_grad = False

for param in model.fc.parameters():
    param.requires_grad = True

total_params = sum(param.numel() for param in model.parameters())
trainable_params = sum(
    param.numel()
    for param in model.parameters()
    if param.requires_grad
)

print("ResNet50 loaded with ImageNet pretrained weights.")
print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

print("Device:", device)


In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.fc.parameters(),
    lr=0.001
)

print("Loss function and optimizer created.")


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


def validate_one_epoch(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    return running_loss / total, correct / total

print("Training and validation functions created.")


In [ ]:
NUM_EPOCHS = 3

best_val_acc = 0.0
best_model_path = "/content/drive/MyDrive/resnet50_imagenet_best.pth"

train_losses = []
train_accuracies = []
valid_losses = []
valid_accuracies = []

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    valid_loss, valid_acc = validate_one_epoch(
        model,
        valid_loader,
        criterion,
        device
    )

    train_losses.append(train_loss)
    train_accuracies.append(train_acc)
    valid_losses.append(valid_loss)
    valid_accuracies.append(valid_acc)

    print(
        f"Epoch {epoch + 1}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Valid Loss: {valid_loss:.4f} | "
        f"Valid Acc: {valid_acc:.4f}"
    )

    if valid_acc > best_val_acc:
        best_val_acc = valid_acc
        torch.save(model.state_dict(), best_model_path)

        print(
            f"  ✓ Best model saved! "
            f"Validation accuracy: {valid_acc:.4f}"
        )

print("\nTraining complete.")
print("Best validation accuracy:", best_val_acc)
print("Best model saved at:", best_model_path)


In [ ]:
model.load_state_dict(
    torch.load(best_model_path, map_location=device)
)

model = model.to(device)
model.eval()

print("Best model loaded successfully.")
print("Best validation accuracy:", best_val_acc)


In [ ]:
model.eval()

all_labels = []
all_predictions = []
all_probabilities = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        probabilities = torch.softmax(outputs, dim=1)
        predictions = outputs.argmax(dim=1)

        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(predictions.cpu().numpy())
        all_probabilities.extend(probabilities[:, 1].cpu().numpy())

y_true = np.array(all_labels)
y_pred = np.array(all_predictions)
y_prob = np.array(all_probabilities)

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_prob)

tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
specificity = tn / (tn + fp)

print("FINAL TEST RESULTS")
print("------------------")
print(f"Accuracy    : {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Precision   : {precision:.4f} ({precision*100:.2f}%)")
print(f"Recall      : {recall:.4f} ({recall*100:.2f}%)")
print(f"Specificity : {specificity:.4f} ({specificity*100:.2f}%)")
print(f"F1 Score    : {f1:.4f} ({f1*100:.2f}%)")
print(f"ROC-AUC     : {auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClass meaning:")
print("0 = Normal")
print("1 = Cancer/Tumor")


## Training and Validation Curves


In [ ]:
epochs = range(1, NUM_EPOCHS + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs, train_accuracies, marker="o", label="Train Accuracy")
plt.plot(epochs, valid_accuracies, marker="o", label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("ResNet50 ImageNet — Accuracy")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(epochs, train_losses, marker="o", label="Train Loss")
plt.plot(epochs, valid_losses, marker="o", label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("ResNet50 ImageNet — Loss")
plt.legend()
plt.grid(True)
plt.show()


## ROC Curve


In [ ]:
fpr, tpr, _ = roc_curve(y_true, y_prob)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, label=f"ROC-AUC = {auc:.4f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ResNet50 ImageNet — ROC Curve")
plt.legend()
plt.grid(True)
plt.show()


## Final Test Performance

| Metric | Result |
|---|---:|
| Accuracy | 88.75% |
| Precision | 95.14% |
| Recall / Sensitivity | 75.06% |
| Specificity | 97.54% |
| F1 Score | 83.91% |
| ROC-AUC | 94.73% |

**Confusion matrix:** `[[634, 16], [104, 313]]`

The model is presented as an AI-assisted screening/decision-support prototype, not as a clinical diagnostic system.
